In [8]:
from scipy.ndimage import median_filter

def estimate_bp(p_beats, amp, k_sys=0.5, k_dia=0.7, edge=3):
    n = len(amp)
    if n < 2*edge + 4:
        return dict(sbp=None, dbp=None, map=None, warn='мало ударов')
    sm = median_filter(amp, size=5, mode='nearest')

    # максимум ищем ТОЛЬКО во внутренней части: края испорчены накачкой/фильтрами
    im = edge + int(np.argmax(sm[edge:n-edge]))
    amax = sm[im]
    warn = []
    if im <= edge or im >= n - edge - 1:
        warn.append('максимум огибающей на краю записи')

    def cross(i_from, i_to, level, rising):
        seg = sm[i_from:i_to]
        hit = np.where(seg >= level)[0] if rising else np.where(seg <= level)[0]
        if len(hit) == 0:
            return None
        j = i_from + hit[0]
        if j == i_from:
            return p_beats[j]
        a0, a1 = sm[j-1], sm[j]
        w = (level - a0)/(a1 - a0) if a1 != a0 else 0.0
        return p_beats[j-1] + w*(p_beats[j] - p_beats[j-1])

    # СИСТОЛА: берём ПОСЛЕДНЮЮ точку ниже порога перед максимумом,
    # затем первое пересечение после неё. Выброс на старте больше не ломает поиск.
    thr = k_sys*amax
    below = np.where(sm[:im] < thr)[0]
    if len(below) == 0:
        sbp = None
        warn.append('огибающая сразу выше порога систолы (артефакт начала спуска)')
    else:
        sbp = cross(below[-1], im+1, thr, rising=True)

    # ДИАСТОЛА: первое пересечение после максимума
    dbp = cross(im, n, k_dia*amax, rising=False)
    if dbp is None:
        warn.append(f'спад не достиг {k_dia:.0%} максимума до конца записи')

    return dict(sbp=sbp, dbp=dbp, map=p_beats[im], warn='; '.join(warn) or 'OK')